# 07 · Lineage check

The job's last task (#33). Confirms, from Unity Catalog lineage, that:
- every gold table reads only from `hawkerreach.silver` and `hawkerreach.gold`
- the app's service principal reads only `hawkerreach.gold`

It logs what it finds to `data_quality_log` and never fails the run. If system tables aren't available, it falls back to a static check that every app query file references only `hawkerreach.gold.*`; the static check runs either way.

In [0]:
import glob, json, os, re
from pyspark.sql import functions as F
from dq import DQLog  # dq.py at the repo root

CATALOG= "hawkerreach"
dbutils.widgets.text("app_name", "hawkerreach")
dbutils.widgets.text("lookback_days", "30")
dbutils.widgets.text("log_schema", "silver")  # dev_<name> for test runs, so they don't write to the shared log
APP= dbutils.widgets.get("app_name")
DAYS= int(dbutils.widgets.get("lookback_days"))
ALLOWED_GOLD_SOURCES= {f"{CATALOG}.silver", f"{CATALOG}.gold"}
APP_SCHEMA= f"{CATALOG}.gold"
dq= DQLog(spark, "07_lineage_check", schema=f"{CATALOG}.{dbutils.widgets.get('log_schema')}")
WINDOW= f"lineage from the last {DAYS} days, not only this run"
summary= {"window_days": DAYS}

def record(check, n_checked, bad, detail):
    """Logs each problem and one summary row to data_quality_log; bad is a list of (entity, problem)."""
    dq.issues(check, [(k, f"{d} ({WINDOW})") for k, d in bad])
    dq.step(check, n_checked, n_checked - len({k for k, _ in bad}), f"{detail} ({WINDOW})")
    summary[check]= {"checked": n_checked, "problems": [{"entity": k, "problem": d} for k, d in bad]}

In [0]:
# 1. gold tables read only from silver and gold
try:
    lineage= spark.table("system.access.table_lineage").filter(F.col("event_date") >= F.date_sub(F.current_date(), DAYS))
    gold_reads= (lineage.filter((F.col("target_table_catalog") == CATALOG) & (F.col("target_table_schema") == "gold")
                                & F.col("source_table_full_name").isNotNull())
                 .select("target_table_full_name", "source_table_full_name").distinct())
    source_schema= F.regexp_extract("source_table_full_name", r"^([^.]+\.[^.]+)\.", 1)  # catalog.schema
    bad= [(r["target_table_full_name"], f"reads {r['source_table_full_name']}")
          for r in gold_reads.filter(~source_schema.isin(*ALLOWED_GOLD_SOURCES)).collect()]
    record("lineage_gold_sources", gold_reads.count(), bad, "gold table <- source pairs that read only silver/gold")
    system_tables= True
except Exception as e:  # e.g. no system tables on this workspace
    print("system.access.table_lineage not available:", str(e)[:200])
    dq.issues("lineage_unavailable", [("system.access.table_lineage", str(e)[:500])])
    system_tables= False
summary["system_tables"]= system_tables

In [0]:
# 2. the app's service principal reads only gold
if system_tables:
    try:
        from databricks.sdk import WorkspaceClient
        sp= WorkspaceClient().apps.get(APP).service_principal_client_id
    except Exception as e:
        sp= None
        print(f"couldn't look up app {APP}:", str(e)[:200])
        dq.issues("lineage_app_lookup", [(APP, str(e)[:500])])
    if sp:
        reads= [r["source_table_full_name"] for r in
                lineage.filter((F.col("created_by") == sp) & F.col("source_table_full_name").isNotNull())
                .select("source_table_full_name").distinct().collect()]
        bad= [(t, f"app {APP} ({sp}) read a non-gold table") for t in reads if not t.startswith(APP_SCHEMA + ".")]
        record("lineage_app_reads", len(reads), bad, f"tables app {APP} read that are gold")
        summary["lineage_app_reads"]["tables"]= sorted(reads)

In [0]:
# 3. static fallback, run either way: every app query file names only hawkerreach.gold tables.
# Every name after FROM or JOIN counts, so a table without its catalog (gold.x, or block_access) is
# flagged too; names a WITH clause defines are skipped.
files= sorted(glob.glob(os.path.join(os.getcwd(), "app", "config", "queries", "*.sql")))
TABLE_REF= re.compile(r"\b(?:from|join)\s+([a-z_][\w.]*)", re.I)
CTE= re.compile(r"\b([a-z_]\w*)\s+as\s*\(", re.I)
bad= []
for f in files:
    sql= re.sub(r"--[^\n]*", "", open(f).read())  # comments may name other tables
    ctes= {c.lower() for c in CTE.findall(sql)}
    for t in TABLE_REF.findall(sql):
        if t.lower() not in ctes and not t.lower().startswith(APP_SCHEMA + "."):
            bad.append((os.path.basename(f), f"references {t}"))
if files:
    record("lineage_static_queries", len(files), bad, "app query files that name only hawkerreach.gold tables")
else:
    print("no app query files found next to this notebook; the static check didn't run")
    dq.issues("lineage_static_queries", [("app/config/queries", "no query files synced with the job")])

In [0]:
problems= sum(len(v["problems"]) for v in summary.values() if isinstance(v, dict) and "problems" in v)
print("lineage check:", "OK" if problems == 0 else f"{problems} problem(s), logged to data_quality_log")
print(json.dumps(summary, indent=1))
dbutils.notebook.exit(json.dumps(summary))